Profiling GPU usage in PyTorch

The CPU profiler cannot track GPU memory usage directly. However, PyTorch provides specialized APIs for profiling GPU usage. These APIs can help you identify bottlenecks and optimize your GPU computations.

In [ ]:
print("GPU Kernal Ready")

GPU Kernal Is Ready


In [2]:
import torch

torch.cuda.memory_allocated()


Caching the list of root modules, please wait!
(This will only be done once - type '%rehashx' to reset cache!)



0

In [3]:
torch.cuda.memory_reserved()

0

In [5]:
# Reserve GPU memory
torch.cuda.empty_cache()

# Get available memory
print("Allocated:", torch.cuda.memory_allocated())
print("Max Allocated:", torch.cuda.max_memory_allocated())
print("Reserved:", torch.cuda.memory_reserved())
print("Max Reserved:", torch.cuda.max_memory_reserved())

# Allocate GPU memory
tensor = torch.randn(1000, 1000).cuda()

# Get available memory after allocation
print("Allocated:", torch.cuda.memory_allocated())
print("Max Allocated:", torch.cuda.max_memory_allocated())
print("Reserved:", torch.cuda.memory_reserved())
print("Max Reserved:", torch.cuda.max_memory_reserved())


Allocated: 4000256
Max Allocated: 4000256
Reserved: 20971520
Max Reserved: 20971520
Allocated: 4000256
Max Allocated: 8000512
Reserved: 20971520
Max Reserved: 20971520


In [6]:
print(torch.cuda.memory_summary())

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   3906 KiB |   7813 KiB |   7813 KiB |   3906 KiB |
|       from large pool |   3906 KiB |   7813 KiB |   7813 KiB |   3906 KiB |
|       from small pool |      0 KiB |      0 KiB |      0 KiB |      0 KiB |
|---------------------------------------------------------------------------|
| Active memory         |   3906 KiB |   7813 KiB |   7813 KiB |   3906 KiB |
|       from large pool |   3906 KiB |   7813 KiB |   7813 KiB |

In [7]:
def detect_nan(model, loss, step):
    if torch.isnan(loss):
        print(f"NaN loss at step {step}")
        for name, param in model.named_parameters():
            if param.grad is not None:
                if torch.isnan(param.grad).any():
                    print(f"  NaN gradient in {name}")
                if torch.isinf(param.grad).any():
                    print(f"  Inf gradient in {name}")
        return True
    return False

In [8]:
def check_data_leakage(train_set, test_set, id_column="id"):
    train_ids = set(train_set[id_column].tolist())
    test_ids = set(test_set[id_column].tolist())
    overlap = train_ids & test_ids
    if overlap:
        print(f"DATA LEAKAGE: {len(overlap)} samples in both train and test")
        return True
    return False

In [9]:
def check_devices(model, *tensors):
    model_device = next(model.parameters()).device
    print(f"Model device: {model_device}")
    for i, t in enumerate(tensors):
        if t.device != model_device:
            print(f"  WARNING: tensor {i} on {t.device}, model on {model_device}")

In [10]:
!pip install tensorboard

In [11]:
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter("runs/experiment_1")

for step in range(num_steps):
    loss = train_step(model, batch)

    writer.add_scalar("loss/train", loss.item(), step)
    writer.add_scalar("lr", optimizer.param_groups[0]["lr"], step)

    if step % 100 == 0:
        for name, param in model.named_parameters():
            writer.add_histogram(f"weights/{name}", param, step)
            if param.grad is not None:
                writer.add_histogram(f"grads/{name}", param.grad, step)

writer.close()

NameError: name 'num_steps' is not defined